# Model LP2

Model LP2 extends [Model LP](GL06LP.ipynb) from Chapter 5 of {cite:t}`GodleyLavoie2006MonetaryEconomicsIntegrated` by making bond prices **endogenous** through a target-proportion mechanism. In Model LP, bond prices are set exogenously. In LP2, the Treasury adjusts bond prices when the share of bonds in household government debt moves outside a target range $[\text{bot}, \text{top}]$. Households form **adaptive expectations** about future bond prices.

## Module Contents

As with all `MacroStat` models, GL06LP2 is divided into Variables, Parameters, Scenarios, and Behavior. The module-level documentation can be found at:

```{eval-rst}
.. toctree::
    :maxdepth: 2

    Variables <GL06LP2/variables.rst>
    Parameters <GL06LP2/parameters.rst>
    Equations <GL06LP2/equations.rst>
    Scenarios <GL06LP2/scenarios.rst>
```

## Key Changes from LP

Model LP2 differs from [Model LP](GL06LP.ipynb) in three respects:

### 1. Endogenous Bond Price (Target-Proportion Mechanism)

The Treasury monitors the proportion of bonds in total household government debt:

```{math}
TP(t) = \frac{BL_h(t-1) \cdot p_{bl}(t-1)}{BL_h(t-1) \cdot p_{bl}(t-1) + B_h(t-1)}
```

When $TP > \text{top}$, the Treasury lets bond prices drift upwards; when $TP < \text{bot}$, prices drift downwards:

```{math}
p_{bl}(t) = \left(1 + z_1 \cdot \beta - z_2 \cdot \beta\right) \cdot p_{bl}(t-1)
```

where $z_1 = \mathbb{1}[TP > \text{top}]$ and $z_2 = \mathbb{1}[TP < \text{bot}]$.

### 2. Adaptive Expectations for Bond Prices

Instead of static expectations ($p_{bl}^e = p_{bl}$), households form adaptive expectations:

```{math}
p_{bl}^e(t) = p_{bl}^e(t-1) - \beta_e \cdot \left(p_{bl}^e(t-1) - p_{bl}(t)\right)
```

### 3. New Parameters

| Parameter | Notation | Default | Description |
|-----------|----------|---------|-------------|
| ExpectationAdjustmentSpeed | $\beta_e$ | 0.5 | Speed of adaptive expectations |
| BondPriceAdjustmentStep | $\beta$ | 0.001 | Bond price adjustment step |
| TargetProportionUpper | $\text{top}$ | 0.52 | Upper band for TP |
| TargetProportionLower | $\text{bot}$ | 0.47 | Lower band for TP |

### Transaction Flow Matrix

```{csv-table} Accounting Transaction Matrix for Model LP2
:file: GL06LP2/transaction_matrix.csv
:header-rows: 2
:stub-columns: 1
```

### Balance Sheet Matrix

```{csv-table} Balance Sheet for Model LP2
:file: GL06LP2/balance_sheet.csv
:header-rows: 2
:stub-columns: 1
```

## Model Dynamics

### Preparatory Steps

In [ ]:
%load_ext autoreload
%autoreload 2

import importlib
import logging
import sys

from matplotlib import pyplot as plt
from matplotlib.ticker import PercentFormatter

from macrostat.models.GL06LP2 import GL06LP2, ParametersGL06LP2, ScenariosGL06LP2

plt.style.use("../../macrostat.mplstyle")
importlib.reload(logging)
logging.basicConfig(stream=sys.stdout, level=logging.INFO)

### Convergence to the Steady State

In [ ]:
params = ParametersGL06LP2(
    hyperparameters={"timesteps": 100, "timesteps_initialization": 1}
)
model = GL06LP2(parameters=params)
model.simulate()
output = model.variables.to_pandas()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
fig.suptitle("Figure LP2.1: Baseline Convergence", fontsize=14)

axes[0, 0].plot(output.index, output["NationalIncome"], color="k")
axes[0, 0].set_title("National Income (Y)")

axes[0, 1].plot(output.index, output["ConsumptionHousehold"], color="k", label="C")
axes[0, 1].plot(output.index, output["DisposableIncome"], color="g", linestyle="-.", label="YDr")
axes[0, 1].set_title("Consumption & Disposable Income")
axes[0, 1].legend()

axes[0, 2].plot(output.index, output["Wealth"], color="k")
axes[0, 2].set_title("Household Wealth (V)")

axes[1, 0].plot(output.index, output["BondPrice"], color="k")
axes[1, 0].set_title("Bond Price ($p_{bl}$)")

axes[1, 1].plot(output.index, output["TargetProportion"], color="k")
axes[1, 1].axhline(0.52, color="r", linestyle=":", alpha=0.5, label="top")
axes[1, 1].axhline(0.47, color="b", linestyle=":", alpha=0.5, label="bot")
axes[1, 1].set_title("Target Proportion (TP)")
axes[1, 1].legend()

axes[1, 2].plot(output.index, output["HouseholdCashStock"], label="Cash")
axes[1, 2].plot(output.index, output["HouseholdBillStock"], label="Bills")
axes[1, 2].plot(
    output.index,
    output["HouseholdBondStock"] * output["BondPrice"],
    label="Bonds (value)",
)
axes[1, 2].set_title("Household Portfolio")
axes[1, 2].legend()

plt.tight_layout()
plt.show()

### Perturbation 1: Rise in the Bill Rate

Following Section 5.8 of {cite:t}`GodleyLavoie2006MonetaryEconomicsIntegrated`, we increase the interest rate on bills from 3% to 4% and observe how the endogenous bond price responds.

In [ ]:
scenarios = ScenariosGL06LP2(parameters=params)
sc1 = scenarios.get_scenario_index("Scenario.1: Rise in bill rate")
model1 = GL06LP2(parameters=params, scenarios=scenarios)
model1.simulate(scenario=sc1)
output_sc1 = model1.variables.to_pandas()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
fig.suptitle("Figure LP2.2: Rise in Bill Rate (rb: 0.03 → 0.04)", fontsize=14)

for ax in axes.flat:
    ax.set_xlabel("Period")

axes[0, 0].plot(output.index, output["NationalIncome"], "k--", label="Baseline")
axes[0, 0].plot(output_sc1.index, output_sc1["NationalIncome"], "k-", label="Shock")
axes[0, 0].set_title("National Income (Y)")
axes[0, 0].legend()

axes[0, 1].plot(output.index, output["Wealth"], "k--", label="Baseline")
axes[0, 1].plot(output_sc1.index, output_sc1["Wealth"], "k-", label="Shock")
axes[0, 1].set_title("Household Wealth (V)")
axes[0, 1].legend()

axes[0, 2].plot(output.index, output["ConsumptionHousehold"], "k--", label="Baseline")
axes[0, 2].plot(output_sc1.index, output_sc1["ConsumptionHousehold"], "k-", label="Shock")
axes[0, 2].set_title("Consumption (C)")
axes[0, 2].legend()

axes[1, 0].plot(output.index, output["BondPrice"], "k--", label="Baseline")
axes[1, 0].plot(output_sc1.index, output_sc1["BondPrice"], "k-", label="Shock")
axes[1, 0].set_title("Bond Price ($p_{bl}$)")
axes[1, 0].legend()

axes[1, 1].plot(output.index, output["TargetProportion"], "k--", label="Baseline")
axes[1, 1].plot(output_sc1.index, output_sc1["TargetProportion"], "k-", label="Shock")
axes[1, 1].axhline(0.52, color="r", linestyle=":", alpha=0.5)
axes[1, 1].axhline(0.47, color="b", linestyle=":", alpha=0.5)
axes[1, 1].set_title("Target Proportion (TP)")
axes[1, 1].legend()

axes[1, 2].plot(output.index, output["CapitalGains"], "k--", label="Baseline")
axes[1, 2].plot(output_sc1.index, output_sc1["CapitalGains"], "k-", label="Shock")
axes[1, 2].set_title("Capital Gains (CG)")
axes[1, 2].legend()

plt.tight_layout()
plt.show()

### Perturbation 2: Expected Bond Price Fall

In this scenario, households' expected bond price drops by 1 unit. This shifts portfolio demand away from bonds toward bills and cash, triggering adjustments through the target-proportion mechanism.

In [ ]:
sc2 = scenarios.get_scenario_index("Scenario.2: Expected bond price fall")
model2 = GL06LP2(parameters=params, scenarios=scenarios)
model2.simulate(scenario=sc2)
output_sc2 = model2.variables.to_pandas()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
fig.suptitle("Figure LP2.3: Expected Bond Price Fall", fontsize=14)

for ax in axes.flat:
    ax.set_xlabel("Period")

axes[0, 0].plot(output.index, output["NationalIncome"], "k--", label="Baseline")
axes[0, 0].plot(output_sc2.index, output_sc2["NationalIncome"], "k-", label="Shock")
axes[0, 0].set_title("National Income (Y)")
axes[0, 0].legend()

axes[0, 1].plot(output.index, output["Wealth"], "k--", label="Baseline")
axes[0, 1].plot(output_sc2.index, output_sc2["Wealth"], "k-", label="Shock")
axes[0, 1].set_title("Household Wealth (V)")
axes[0, 1].legend()

axes[0, 2].plot(output.index, output["ExpectedBondPrice"], "k--", label="Baseline")
axes[0, 2].plot(output_sc2.index, output_sc2["ExpectedBondPrice"], "k-", label="Shock")
axes[0, 2].set_title("Expected Bond Price ($p_{bl}^e$)")
axes[0, 2].legend()

axes[1, 0].plot(output.index, output["BondPrice"], "k--", label="Baseline")
axes[1, 0].plot(output_sc2.index, output_sc2["BondPrice"], "k-", label="Shock")
axes[1, 0].set_title("Bond Price ($p_{bl}$)")
axes[1, 0].legend()

axes[1, 1].plot(output.index, output["TargetProportion"], "k--", label="Baseline")
axes[1, 1].plot(output_sc2.index, output_sc2["TargetProportion"], "k-", label="Shock")
axes[1, 1].axhline(0.52, color="r", linestyle=":", alpha=0.5)
axes[1, 1].axhline(0.47, color="b", linestyle=":", alpha=0.5)
axes[1, 1].set_title("Target Proportion (TP)")
axes[1, 1].legend()

axes[1, 2].plot(output.index, output["ExpectedReturnOnBonds"], "k--", label="Baseline")
axes[1, 2].plot(output_sc2.index, output_sc2["ExpectedReturnOnBonds"], "k-", label="Shock")
axes[1, 2].set_title("Expected Return on Bonds ($ERbl$)")
axes[1, 2].legend()

plt.tight_layout()
plt.show()